# Paired protocol: CUB-200-2011 (Tables 1 and 2)

DA351 Phase 1, FVLC replication (Lai et al., ICLR 2024). This is the notebook we actually ran on **Kaggle** (GPU T4), with its outputs.

The repo was attached as a Kaggle dataset (Add Input). The results zip it produced is `results_paired/cub/` in this repo.
Only Kaggle's default boilerplate cell was removed, and pip progress bars were trimmed from the output; nothing else was changed.


In [2]:
import os, glob, shutil
os.makedirs('/tmp/code', exist_ok=True)
z = glob.glob('/kaggle/input/**/*fvlc-replication*.zip', recursive=True)
if z:
    !unzip -q -o "{z[0]}" -d /tmp/code/
else:
    src = os.path.dirname(glob.glob('/kaggle/input/**/run_experiments.py', recursive=True)[0])
    shutil.copytree(src, '/tmp/code/fvlc-replication', dirs_exist_ok=True)
%cd /tmp/code/fvlc-replication
!pip install -q -r requirements.txt
os.environ['FVLC_ACT_DIR'] = '/tmp/activations'
OUT = '/kaggle/working/results_paired'
!grep -c "def evaluate_paired" fvlc/pipeline.py
!python tests/test_core.py

/tmp/code/fvlc-replication
  Preparing metadata (setup.py) ... done
1
100%|██████████████████████████████████████████| 50/50 [00:00<00:00, 102.59it/s]
did not converge at 50 iterations (criteria False)
(0) lambda 0.0007, loss 0.0270, acc 1.0000 [val acc 1.0000] [test acc -1.0000], sparsity 0.9791666666666666 [47/48], time 0.5014920234680176, lr 0.1000
universal: TCPC base 1.338e-01 -> FVLC 9.998e-03 | TOPC 5.930e-02 -> 4.852e-03
per_sample: TCPC base 1.338e-01 -> FVLC 1.890e-02 | TOPC 5.930e-02 -> 8.786e-03
all tests passed


In [3]:
!bash scripts/download_cub.sh
!python run_experiments.py --dataset cub --protocol paired --fvlc_init gaussian --out $OUT
!python make_tables.py --results $OUT

CUB_200_2011.tgz    100%[===================>]   1.07G  23.9MB/s    in 50s     
{'train': 5990, 'test': 5790}
CUB ready at data/CUB
Dataset cub | device cuda | seed 42
100%|███████████████████████████████████████| 335M/335M [00:07<00:00, 47.5MiB/s]
[vanilla] 370 concepts -> 331 after CLIP cutoff                                 
  projection: best step 200, val similarity 0.4912
[vanilla] 208 concepts after interpretability cutoff
 79%|██████████████████████████████▊        | 3943/5000 [05:48<01:33, 11.31it/s]
(0) lambda 0.0002, loss 0.3317, acc 1.0000 [val acc 0.7371] [test acc -1.0000], sparsity 0.15454326923076922 [6429/41600], time 348.7245144844055, lr 0.1000
[vanilla] test acc 0.7371  (6429/41600 non-zero final weights)
 14%|█████▌                                  | 692/5000 [01:15<07:48,  9.20it/s]
(0) lambda 0.0000, loss 0.0112, acc 1.0000 [val acc 0.7592] [test acc -1.0000], sparsity 0.064970703125 [6653/102400], time 75.3410472869873, lr 0.1000
[standard] test acc 0.7592

====

In [4]:
!cd /kaggle/working && zip -qr results_cub.zip results_paired -x "*/layers/*" "*/vanilla.pt"
print(open(f'{OUT}/tables.md').read())

# Replication results vs. paper

Ours = this repo (seed 42, evaluation protocol: paired). Paper = Lai et al., ICLR 2024. Accuracy in %.


## Headline numbers (averaged over the 7 paper perturbations)

| Dataset | LF-CBM acc ours / paper | FVLC acc ours / paper | base TCPC ours / paper | FVLC TCPC ours / paper | base TOPC ours / paper | FVLC TOPC ours / paper | seed-only TCPC ours |
|---|---|---|---|---|---|---|---|
| CUB | 73.71 / 74.23 | 72.94 / 74.09 | 4.49e-01 / 1.70e-01 | 2.73e-03 / 2.60e-02 | 3.59e-01 / 2.57e-01 | 1.76e-03 / 1.30e-02 | 3.24e-01 |

## Table 1 – Accuracy

| Method | CUB ours | CUB paper |
|---|---|---|
| Standard (no interpretability) | 75.92 | 76.70 |
| Label-free CBM | 73.71 | 74.23 |
| WP1(5%) – base | 69.74 | 74.08 |
| WP1(5%) – FVLC | 72.94 | 73.96 |
| WP1(10%) – base | 66.72 | 73.97 |
| WP1(10%) – FVLC | 72.94 | 73.92 |
| WP2 – base | 72.35 | 73.96 |
| WP2 – FVLC | 72.94 | 74.44 |
| IP – base | 72.28 | 74.39 |
| IP – FVLC | 72.94 | 74.01 |
| WP1(5%)+WP2 – base

In [5]:
from IPython.display import FileLink, display
import os
os.chdir('/kaggle/working')
print(os.listdir('.'))
if not os.path.exists('results_cub.zip'):
    !zip -qr results_cifar10.zip results_paired -x "*/layers/*" "*/vanilla.pt"
display(FileLink('results_cub.zip'))

['.virtual_documents', 'results_paired', 'results_cub.zip']


/kaggle/working/results_cub.zip